# Stage 1 Only Inference 教程 - 稀疏结构生成与蒸馏加速

本教程演示如何使用 `Stage1OnlyInference` 类：
1. **只加载 Stage 1 相关模型** - 节省约 50% 显存
2. **使用蒸馏加速 4 步推理** - 相比标准 25 步提升约 6 倍速度
3. **可视化稀疏点云坐标** - 大致几何形状
4. **可视化形状潜在特征** - 形状的压缩表示

## 背景

SAM 3D 的两阶段生成流程：
- **Stage 1**: 稀疏结构生成 → 64³ 体素网格中的占据坐标
- **Stage 2**: 结构化潜在生成 → Gaussian Splat 详细特征

如果只需要大致几何形状，可以只运行 Stage 1，显著减少计算量。

## 1. 导入和模型加载

In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

import os
import sys
import time
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

# 添加 notebook 目录到路径
sys.path.insert(0, os.getcwd())

from stage1_inference import (
    Stage1OnlyInference,
    load_image,
    load_single_mask,
    load_masks,
    display_image,
    visualize_sparse_coords,
    visualize_shape_features,
    compare_sparse_coords,
)

In [ ]:
# 设置路径
PATH = os.getcwd()
TAG = "hf"
config_path = f"{PATH}/../checkpoints/{TAG}/pipeline.yaml"

print(f"配置文件路径: {config_path}")
print(f"配置文件存在: {os.path.exists(config_path)}")

if torch.cuda.is_available():
    print(f"\nCUDA 可用: True")
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n警告: CUDA 不可用，推理将在 CPU 上运行（非常慢）")

### 1.1 加载 Stage 1 Only 模型

`Stage1OnlyInference` 类只加载以下模型：
- `ss_generator`: 稀疏结构生成器（Flow Matching）
- `ss_decoder`: VAE 解码器（潜在表示 → 体素网格）
- `ss_condition_embedder`: DINO 条件嵌入器
- `ss_preprocessor`: 图像预处理器
- `pose_decoder`: 姿态解码器

不加载 Stage 2 模型（`slat_generator`, `slat_decoder_gs/mesh` 等），可节省约 50% 显存。

In [ ]:
# 加载 Stage 1 Only 模型
print("正在加载 Stage 1 模型...")
print("(只加载稀疏结构生成相关模型，节省显存)")

# 记录显存使用
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    initial_memory = torch.cuda.memory_allocated() / 1e9

start_time = time.time()
inference = Stage1OnlyInference(config_path, compile=False)
load_time = time.time() - start_time

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated() / 1e9
    print(f"\n模型加载完成！")
    print(f"加载时间: {load_time:.2f} 秒")
    print(f"峰值显存: {peak_memory:.2f} GB")
else:
    print(f"\n模型加载完成！耗时: {load_time:.2f} 秒")

In [ ]:
# 查看加载的模型组件
print("=== 已加载的模型 ===")
for key in inference.models.keys():
    model = inference.models[key]
    num_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  - {key}: {type(model).__name__} ({num_params:.1f}M 参数)")

print("\n=== 已加载的条件嵌入器 ===")
for key in inference.condition_embedders.keys():
    print(f"  - {key}")

## 2. 加载输入图像和掩码

In [ ]:
# 加载示例图像
IMAGE_PATH = f"{PATH}/images/shutterstock_stylish_kidsroom_1640806567/image.png"
IMAGE_NAME = os.path.basename(os.path.dirname(IMAGE_PATH))

image = load_image(IMAGE_PATH)
mask = load_single_mask(os.path.dirname(IMAGE_PATH), index=14)

print(f"图像尺寸: {image.shape}")
print(f"掩码尺寸: {mask.shape}")
print(f"掩码像素数: {mask.sum()}")

display_image(image, masks=[mask])

## 3. 运行 Stage 1 推理

### 3.1 蒸馏加速模式 (4 步)

蒸馏模式基于 **Shortcut Models** 论文，通过训练时的 Self-Consistency Loss 将推理步数从 25 减少到 4。

| 模式 | 步数 | CFG 强度 | 速度提升 |
|------|------|----------|----------||
| 标准 | 25 | 7.0 | 1x |
| 蒸馏 | 4 | 0 (已蒸馏) | ~6x |

In [ ]:
# 蒸馏模式：4 步推理
print("=== 蒸馏模式 (4 步) ===")
print("参数: no_shortcut=False, CFG strength=0 (已蒸馏到模型中)")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()

output_distill = inference.run(
    image,
    mask,
    seed=42,
    steps=4,
    use_distillation=True,
)

distill_time = time.time() - start_time

if torch.cuda.is_available():
    distill_memory = torch.cuda.max_memory_allocated() / 1e9
    print(f"\n推理时间: {distill_time:.2f} 秒")
    print(f"峰值显存: {distill_memory:.2f} GB")
else:
    print(f"\n推理时间: {distill_time:.2f} 秒")

In [ ]:
# 查看输出结构
print("=== 输出结构 ===")
for key, value in output_distill.items():
    if isinstance(value, torch.Tensor):
        print(f"  - {key}: {value.shape} ({value.dtype})")
    elif isinstance(value, np.ndarray):
        print(f"  - {key}: {value.shape} ({value.dtype})")
    else:
        print(f"  - {key}: {type(value).__name__}")

### 3.2 标准模式对比 (25 步)

In [ ]:
# 标准模式：25 步推理
print("=== 标准模式 (25 步) ===")
print("参数: no_shortcut=True, CFG strength=7.0")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()

output_standard = inference.run(
    image,
    mask,
    seed=42,
    steps=25,
    use_distillation=False,
)

standard_time = time.time() - start_time

if torch.cuda.is_available():
    standard_memory = torch.cuda.max_memory_allocated() / 1e9
    print(f"\n推理时间: {standard_time:.2f} 秒")
    print(f"峰值显存: {standard_memory:.2f} GB")
    print(f"\n加速比: {standard_time / distill_time:.1f}x")
else:
    print(f"\n推理时间: {standard_time:.2f} 秒")
    print(f"\n加速比: {standard_time / distill_time:.1f}x")

## 4. 可视化稀疏点云坐标

稀疏点云坐标表示 64³ 体素网格中占据的位置，是大致几何形状的表示。

In [ ]:
# 提取稀疏点云坐标
voxel_distill = output_distill["voxel"].cpu().numpy()
voxel_standard = output_standard["voxel"].cpu().numpy()

print(f"蒸馏模式点云:")
print(f"  - 点数: {len(voxel_distill)}")
print(f"  - 范围: [{voxel_distill.min():.3f}, {voxel_distill.max():.3f}]")
print(f"  - 中心: {voxel_distill.mean(axis=0)}")

print(f"\n标准模式点云:")
print(f"  - 点数: {len(voxel_standard)}")
print(f"  - 范围: [{voxel_standard.min():.3f}, {voxel_standard.max():.3f}]")
print(f"  - 中心: {voxel_standard.mean(axis=0)}")

In [ ]:
# 单独可视化蒸馏模式点云
fig = visualize_sparse_coords(
    voxel_distill,
    title="Sparse Point Cloud (Distillation, 4 steps)",
    color="blue",
    alpha=0.6,
    s=2,
)
plt.tight_layout()
plt.show()

In [ ]:
# 对比可视化：蒸馏 vs 标准
fig = compare_sparse_coords(
    [voxel_standard, voxel_distill],
    labels=["Standard (25 steps)", "Distillation (4 steps)"],
    colors=["green", "blue"],
    figsize=(14, 6),
)
plt.suptitle(f"Speed Improvement: {standard_time/distill_time:.1f}x", fontsize=14)
plt.tight_layout()
plt.show()

## 5. 可视化形状潜在特征

形状潜在特征是 VAE 编码的压缩表示，形状为 `(B, 8, 16, 16, 16)`。

我们使用 PCA 降维到 3D 进行可视化。

In [ ]:
# 提取形状潜在特征
shape_distill = output_distill["shape"].cpu().numpy()
shape_standard = output_standard["shape"].cpu().numpy()

print(f"形状潜在特征:")
print(f"  - 形状: {shape_distill.shape}")
print(f"  - 含义: (Batch, Channels, Depth, Height, Width)")
print(f"  - 压缩比: 64³ 体素 → 8×16³ 潜在")

In [ ]:
# 可视化蒸馏模式的形状潜在特征
fig = visualize_shape_features(
    shape_distill,
    method="pca",
    title="Shape Latent Features (Distillation, 4 steps)",
)
plt.show()

In [ ]:
# 可视化标准模式的形状潜在特征
fig = visualize_shape_features(
    shape_standard,
    method="pca",
    title="Shape Latent Features (Standard, 25 steps)",
)
plt.show()

## 6. 不同步数的蒸馏模式对比

In [ ]:
# 测试不同步数
steps_to_test = [2, 4, 8, 12]
results = {}

print("=== 不同步数的蒸馏模式对比 ===\n")

for steps in steps_to_test:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    
    start_time = time.time()
    
    output = inference.run(
        image,
        mask,
        seed=42,
        steps=steps,
        use_distillation=True,
    )
    
    elapsed = time.time() - start_time
    memory = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    num_points = output["voxel"].shape[0]
    
    results[steps] = {
        'time': elapsed,
        'memory': memory,
        'output': output,
        'num_points': num_points,
    }
    
    print(f"步数: {steps:2d} | 时间: {elapsed:.2f}s | 点数: {num_points}")

In [ ]:
# 可视化不同步数的点云对比
voxel_list = [results[s]['output']['voxel'].cpu().numpy() for s in steps_to_test]
labels = [f"{s} steps" for s in steps_to_test]

fig = compare_sparse_coords(voxel_list, labels, figsize=(16, 8))
plt.suptitle("Distillation Mode: Different Step Counts", fontsize=14)
plt.tight_layout()
plt.show()

## 7. 性能总结

In [ ]:
# 绘制性能对比图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 时间对比
steps_list = [25] + steps_to_test
times = [standard_time] + [results[s]['time'] for s in steps_to_test]
colors = ['#2ecc71'] + ['#3498db'] * len(steps_to_test)

ax1 = axes[0]
bars = ax1.bar(range(len(steps_list)), times, color=colors)
ax1.set_xticks(range(len(steps_list)))
ax1.set_xticklabels([f'Standard\n25 steps'] + [f'Distill\n{s} steps' for s in steps_to_test])
ax1.set_ylabel('Inference Time (seconds)', fontsize=12)
ax1.set_title('Inference Time Comparison', fontsize=14)

for bar, t in zip(bars, times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{t:.2f}s', ha='center', va='bottom', fontsize=10)

# 加速比
ax2 = axes[1]
speedups = [standard_time / t for t in times]
bars2 = ax2.bar(range(len(steps_list)), speedups, color=colors)
ax2.set_xticks(range(len(steps_list)))
ax2.set_xticklabels([f'Standard\n25 steps'] + [f'Distill\n{s} steps' for s in steps_to_test])
ax2.set_ylabel('Speedup', fontsize=12)
ax2.set_title('Speedup vs Standard Mode', fontsize=14)
ax2.axhline(y=1, color='red', linestyle='--', label='Baseline')

for bar, s in zip(bars2, speedups):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{s:.1f}x', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f"{PATH}/gaussians/single/stage1_performance_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 最终性能总结表
print("\n" + "="*70)
print("Performance Summary - Stage 1 Only Inference")
print("="*70)
print(f"{'Mode':<15} {'Steps':<8} {'Time(s)':<10} {'Speedup':<10} {'Use Case'}")
print("-"*70)
print(f"{'Standard':<15} {25:<8} {standard_time:<10.2f} {'1.0x':<10} {'High quality'}")
for s in steps_to_test:
    speedup = standard_time / results[s]['time']
    use_case = 'Quick preview' if s <= 4 else ('Balanced' if s <= 8 else 'High quality')
    print(f"{'Distillation':<15} {s:<8} {results[s]['time']:<10.2f} {speedup:<10.1f}x {use_case}")
print("="*70)

print("\n内存节省: 只加载 Stage 1 模型，相比完整管道节省约 50% 显存")

## 8. 总结

### 关键要点

1. **Stage 1 Only Inference**:
   - 只加载稀疏结构生成相关模型
   - 节省约 50% 显存
   - 输出稀疏点云坐标和形状潜在特征

2. **蒸馏加速**:
   - 4 步蒸馏推理 ≈ 6x 加速
   - 适合快速预览几何形状
   - CFG 强度已蒸馏到模型中

3. **可视化输出**:
   - `voxel`: 稀疏点云坐标 (N, 3)
   - `shape`: 形状潜在特征 (B, 8, 16, 16, 16)
   - 使用 PCA 降维可视化特征

### 推荐设置

| 场景 | 步数 | 蒸馏模式 |
|------|------|----------||
| 快速预览 | 2-4 | True |
| 平衡模式 | 4-8 | True |
| 高质量 | 12-25 | False |